# `ValuationEngineProductOvernightIndexCompositeCashflow` -- Test Notebook

Exercises `ValuationEngineProductOvernightIndexCompositeCashflow`
(`fixedincomelib/yield_curve/valuation_engine.py`), the product-level engine that wraps
`ValuationEngineAnalyticsCompositeIndex` (the raw compounded-overnight-rate engine) with
notional, spread, and discounting to price a single `ProductOvernightIndexCompositeCashflow`
leg -- the same building block a floating leg of an overnight-index swap is made of.

The anchored index's own business-day/holiday conventions are sourced entirely off
`product.index` (the `OvernightCompositeIndex`, via its `business_day_conv`/`payment_holiday_conv`
properties, which delegate to the underlying `OvernightIndex`'s own native settlement
conventions) -- not off the product's `payment_business_day_convention`/`payment_holiday_convention`,
so the daily compounding schedule is always correct regardless of what payment convention the
product itself was built with.

`get_risk()` is not overridden on this class -- it's inherited from the `ValuationEngineProduct`
base (`fixedincomelib/valuation/valuation_engine.py`), which now provides one shared
implementation for every product engine: `value_.backward()` (if the value carries a live
graph), then harvest via `model_.get_gradient(reset=True)`, then overwrite the caller's
`gradient` array in place. There's no `accumulate` option any more -- `get_risk` always resets/
overwrites; a caller wanting portfolio-level aggregation across trades must sum the individual
results itself.

Covers, against a small synthetic USD SOFR curve (`SOFR-1B` projection, `SOFR-1B-FLAT`
discounting, built with a `BRENT` solver off `SOFR-1B` as its reference):

1. PV of a fully forward-looking cashflow against an independent closed-form (curve-implied
   compounded rate + spread, discounted off the funding curve).
2. `pv01()` against its exact closed form (`sign * notional * tau * DF(payment_date) * 1e-4` --
   exact, not a finite-difference estimate, since PV is affine in `forward_rate_`).
3. `get_risk()` against an independent finite-difference (parallel-bump) estimate, checked
   separately against each curve block (`SOFR-1B` projection and `SOFR-1B-FLAT` discounting) --
   bumping only one component's calibration input at a time, since `SOFR-1B-FLAT` is solved
   (`BRENT`) off `SOFR-1B` as its reference and a full-rebuild bump of `SOFR-1B` alone re-solves
   `SOFR-1B-FLAT` back to its own unchanged target, so it isn't a valid probe of `SOFR-1B-FLAT`'s
   own gradient block (and vice versa).
4. `get_risk()` always resets -- calling it twice into the same gradient array gives the
   identical result both times (an overwrite, never an accumulation), and pre-seeding the array
   with unrelated nonzero values gets those values wiped out rather than added to.
5. Sanity checks on the PV/cash settlement branches (`value_date == payment_date` realizes the
   full settlement as cash; `value_date > payment_date` is fully matured, PV and cash both zero)
   and on the `pay_or_rec` sign convention.

In [1]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import numpy as np
import pandas as pd
import torch

from fixedincomelib import *
from fixedincomelib.yield_curve.valuation_engine import ValuationEngineProductOvernightIndexCompositeCashflow
from fixedincomelib.valuation.valuation_parameters import (
    ValuationParametersCollection,
    AnalyticValParam,
    FundingIndexParameter,
)

print("Fixed Income Library is loaded.")

Fixed Income Library is loaded.


## Build a small USD SOFR yield curve

Two-component curve: `SOFR-1B` (overnight index projection, flat 4.30% IFR) and `SOFR-1B-FLAT`
(the discounting/funding curve, `REFERENCE`d off `SOFR-1B` and calibrated with a flat 0.0% IFR
spread -- i.e. it tracks `SOFR-1B` one-for-one, exactly how OIS discounting is set up in
practice). `USD-SOFR-COMPOUND` (`SOFR-1B`, geometric `COMPOUND`) is already registered in
`static_files/indices.yaml`, so no ad hoc index registration is needed.

In [2]:
bm_list = [
    qfCreateBuildMethod('YC_OVERNIGHT_INDEX_ELEMENT', {
        'TARGET': 'SOFR-1B',
        'INSTANTANEOUS FORWARD RATE': 'USD-SOFR-OIS-1B-IFR',
    }),
    qfCreateBuildMethod('YC_FUNDING_ELEMENT', {
        'TARGET': 'SOFR-1B-FLAT',
        'REFERENCE': 'SOFR-1B',
        'INSTANTANEOUS FORWARD RATE': 'USD-SOFR-OIS-1B-FLAT-IFR',
    }),
    qfCreateBuildMethod('YC_COMMON', {
        'TARGET': 'USD',
        'FUNDING PARAMETERS': 'SOFR-1B-FLAT',
        'SOLVER METHOD': 'BRENT',
    }),
]
build_method_collection = qfCreateModelBuildMethodCollection(bm_list)

data_type = 'INSTANTANEOUS FORWARD RATE'
tenors = ['3M', '6M', '1Y', '2Y', '5Y', '10Y', '30Y']

sofr_ifr = pd.DataFrame(index=tenors)
sofr_ifr['values'] = [0.0430] * len(tenors)  # flat 4.30% IFR curve

flat_ifr = pd.DataFrame(index=['1Y', '10Y', '30Y'])
flat_ifr['values'] = [0.0, 0.0, 0.0]  # zero spread over SOFR-1B

def build_ifr_data(sofr_values=None, flat_values=None):
    sofr_df = sofr_ifr.copy()
    if sofr_values is not None:
        sofr_df['values'] = sofr_values
    flat_df = flat_ifr.copy()
    if flat_values is not None:
        flat_df['values'] = flat_values
    return qfCreateDataCollection([
        qfCreateData1D(data_type, 'USD-SOFR-OIS-1B-IFR', sofr_df),
        qfCreateData1D(data_type, 'USD-SOFR-OIS-1B-FLAT-IFR', flat_df),
    ])

value_date = '2026-07-17'
yc_usd = qfCreateModel(value_date, 'YIELD_CURVE', build_ifr_data(), build_method_collection)

sofr_index = IndexRegistry().get('SOFR-1B')
sofr_composite_index = IndexRegistry().get('USD-SOFR-COMPOUND')
funding_identifier = FundingIdentifierRegistry().get('SOFR-1B-FLAT')

# the engine sources its own daily-compounding-schedule conventions off sofr_composite_index
# (business_day_conv / payment_holiday_conv, delegating to sofr_index's own native settlement
# conventions) -- these are used here purely to build a sensible payment schedule/closed form
# on the *product* side, and happen to be the exact same conventions the engine uses internally.
biz_conv = sofr_index.payment_business_day_conv
hol_conv = sofr_index.settlement_holiday

vpc = ValuationParametersCollection([
    FundingIndexParameter({
        'FUNDING INDEX': 'SOFR-1B-FLAT',
        'CURRENCIES': '',
        'FUNDING INDICES': '',
        'UNDERLYING FUNDING INDEX': '',
    }),
])

print(f'USD SOFR model built with {yc_usd.num_components} components, value date {value_date}.')
print('component_order:', yc_usd.component_order_)

USD SOFR model built with 2 components, value date 2026-07-17.
component_order: ['SOFR-1B', 'SOFR-1B-FLAT']


## Build the product -- a fully forward-looking 3M composite cashflow

`effective_date` is well after `value_date`, so the whole period is curve-implied (no historical
fixings needed) -- this keeps the closed-form PV check simple while still exercising the full
torch-autograd pipeline end to end (`.value` carries a live graph).

In [3]:
effective_date = Date('2026-08-19')
termination_date = add_period(effective_date, Period('3M'), biz_conv, hol_conv)
spread = 0.0015  # 15bp spread over compounded SOFR
notional = 10_000_000.0

product = ProductOvernightIndexCompositeCashflow(
    effective_date,
    TermOrDate(termination_date),
    PayOrReceive.RECEIVE,
    sofr_composite_index,
    spread,
    Currency('USD'),
    notional,
    payment_business_day_convention=biz_conv,
    payment_holiday_convention=hol_conv,
)

print('effective  :', product.effective_date)
print('termination:', product.termination_date)
print('payment    :', product.payment_date)
print('spread     :', product.spread)
print('accrued    :', product.accrued)
assert product.spread == spread, 'product.spread must round-trip the constructor spread argument'

engine = ValuationEngineProductOvernightIndexCompositeCashflow(yc_usd, vpc, product, ValuationRequest.PV)
engine.calculate_value()

print('forward_rate:', engine.forward_rate_)
print('value       :', engine.value)
print('cash        :', engine.cash)

effective  : August 19th, 2026
termination: November 19th, 2026
payment    : November 19th, 2026
spread     : 0.0015
accrued    : 0.25555555555555554
forward_rate: tensor(0.0426, dtype=torch.float64, grad_fn=<DivBackward0>)
value       : tensor(111157.3543, dtype=torch.float64, grad_fn=<MulBackward0>)
cash        : 0.0


## Test 1 -- PV against an independent closed form

`forward_rate_` should equal the curve-implied compounded rate `(DF(effective)/DF(termination) -
1)/tau` off the `SOFR-1B` projection curve; the settlement is `notional * tau * (forward_rate +
spread)`, discounted from `payment_date` to `value_date` off the `SOFR-1B-FLAT` funding curve.
Both legs computed here independently of the engine's own code.

**Gotcha:** `model.discount_factor(index, date)` defaults to `calc_grad=False`, which flips
`requires_grad` to `False` *in place* on the curve component's shared state-data tensor
(`Interpolator1D.values_.requires_grad_(calc_grad)`) -- since that tensor is the same object
reused by every call against this model, a plain (non-`calc_grad`) discount-factor lookup here
would silently break `.backward()`/`get_gradient()` for `engine` later in this notebook (no
error -- the gradient just silently comes back as zero). Always pass `calc_grad=True` when
reusing a model that risk will be computed on afterwards.

In [4]:
tau = product.accrued
df_eff = yc_usd.discount_factor(sofr_index, product.effective_date, calc_grad=True)
df_term = yc_usd.discount_factor(sofr_index, product.termination_date, calc_grad=True)
expected_forward = (df_eff / df_term - 1.0) / tau

df_pay = yc_usd.discount_factor(funding_identifier, product.payment_date, calc_grad=True)
expected_settlement = notional * tau * (expected_forward + spread)
expected_pv = expected_settlement * df_pay

print('engine forward_rate :', float(engine.forward_rate_.detach()))
print('closed-form forward :', float(expected_forward.detach()))
print('engine PV           :', float(engine.value.detach()))
print('closed-form PV      :', float(expected_pv.detach()))

assert abs(float(engine.forward_rate_.detach()) - float(expected_forward.detach())) < 1e-10
assert abs(float(engine.value.detach()) - float(expected_pv.detach())) < 1e-6
assert engine.cash == 0.0, 'nothing settles today: value_date is well before payment_date'
print('Test 1 PASSED (PV matches independent closed form)')

engine forward_rate : 0.04264162403344017
closed-form forward : 0.04264162403344017
engine PV           : 111157.35434079105
closed-form PV      : 111157.35434079105
Test 1 PASSED (PV matches independent closed form)


## Test 2 -- `pv01()` against its exact closed form

PV here is affine in `forward_rate_` (`PV = sign * notional * tau * (forward_rate + spread) *
DF(payment_date)`, and `DF(payment_date)` doesn't itself depend on `forward_rate_`), so `pv01()`
should match the closed form `sign * notional * tau * DF(payment_date) * 1e-4` *exactly* --
no finite-difference tolerance needed.

In [5]:
pv01 = engine.pv01()
expected_pv01 = 1.0 * notional * tau * float(df_pay.detach()) * 1e-4  # sign_ == +1.0 (RECEIVE)

print('engine pv01      :', pv01)
print('closed-form pv01 :', expected_pv01)

assert abs(pv01 - expected_pv01) < 1e-8
print('Test 2 PASSED (pv01 matches exact closed form)')

engine pv01      : 251.819811288733
closed-form pv01 : 251.819811288733
Test 2 PASSED (pv01 matches exact closed form)


## Test 3 -- `get_risk()` against finite difference

`SOFR-1B-FLAT` is `REFERENCE`d off `SOFR-1B` and solved (`BRENT`) to hit its own IFR target (here
a flat 0.0%, i.e. zero spread), so a full-model-rebuild bump of *one* component's target input
only probes *that* component's own gradient block cleanly: bumping `SOFR-1B`'s target re-solves
`SOFR-1B-FLAT` right back to the same zero-spread state (its own target didn't move), so the
resulting finite difference reflects only the `SOFR-1B` block's sensitivity -- and vice versa for
bumping `SOFR-1B-FLAT`'s own target. Each bump is therefore compared against its own block only,
not the full gradient vector.

In [6]:
EPS = 1e-6
REL_TOL = 1e-4

n_state = sum(yc_usd.gradient_lengths_) if yc_usd.gradient_lengths_ else len(tenors) + len(['1Y', '10Y', '30Y'])
sofr_len = len(tenors)

grad = np.zeros(n_state)
engine.get_risk(gradient=grad)
sofr_block = grad[:sofr_len]
flat_block = grad[sofr_len:]
print('SOFR-1B block     :', sofr_block, ' sum:', np.sum(sofr_block))
print('SOFR-1B-FLAT block:', flat_block, ' sum:', np.sum(flat_block))

def bumped_pv(sofr_bump=0.0, flat_bump=0.0):
    dc = build_ifr_data(
        sofr_values=[0.0430 + sofr_bump] * len(tenors),
        flat_values=[0.0 + flat_bump] * 3,
    )
    yc_bumped = qfCreateModel(value_date, 'YIELD_CURVE', dc, build_method_collection)
    eng_bumped = ValuationEngineProductOvernightIndexCompositeCashflow(yc_bumped, vpc, product, ValuationRequest.PV)
    eng_bumped.calculate_value()
    return float(eng_bumped.value.detach())

base_pv = float(engine.value.detach())

fd_sofr = (bumped_pv(sofr_bump=EPS) - base_pv) / EPS
fd_flat = (bumped_pv(flat_bump=EPS) - base_pv) / EPS

print('finite-diff  (bump SOFR-1B)      :', fd_sofr, ' vs analytic block sum:', np.sum(sofr_block))
print('finite-diff  (bump SOFR-1B-FLAT) :', fd_flat, ' vs analytic block sum:', np.sum(flat_block))

assert abs(fd_sofr - np.sum(sofr_block)) / abs(np.sum(sofr_block)) < REL_TOL
assert abs(fd_flat - np.sum(flat_block)) / abs(np.sum(flat_block)) < REL_TOL
print('Test 3 PASSED (analytic risk matches finite difference in both curve blocks)')

SOFR-1B block     : [1636121.46037858  836578.85909159       0.               0.
       0.               0.               0.        ]  sum: 2472700.319470176
SOFR-1B-FLAT block: [-38067.58710301      0.             -0.        ]  sum: -38067.587103010635
finite-diff  (bump SOFR-1B)      : 2472699.78231634  vs analytic block sum: 2472700.319470176
finite-diff  (bump SOFR-1B-FLAT) : -38067.58057908155  vs analytic block sum: -38067.587103010635
Test 3 PASSED (analytic risk matches finite difference in both curve blocks)


## Test 4 -- `get_risk()` always resets (never accumulates)

`get_risk(gradient=...)` writes this trade's own risk into the caller-supplied array on every
call -- it has no `accumulate` option, so a second call must overwrite the array's previous
contents rather than add to them. Confirmed here two ways: (a) calling it twice in a row gives
the identical result both times (not 2x), and (b) pre-seeding the array with unrelated nonzero
values before calling `get_risk` gets those values wiped out, not added to.

In [7]:
grad_a = np.zeros(n_state)
engine.get_risk(gradient=grad_a)
print('first call  grad sum:', np.sum(grad_a))

grad_b = np.zeros(n_state)
engine.get_risk(gradient=grad_b)
print('second call grad sum:', np.sum(grad_b))

assert np.allclose(grad_a, grad_b), 'two independent calls should produce the identical result'
print('two separate calls agree -- get_risk is deterministic/idempotent given the same model state')

# pre-seed with unrelated nonzero values -- get_risk must wipe these out, not add to them
grad_seeded = np.full(n_state, 999.0)
engine.get_risk(gradient=grad_seeded)
assert np.allclose(grad_seeded, grad_a), 'get_risk must overwrite the array, not accumulate into it'
print('pre-seeded nonzero values are correctly overwritten, not accumulated into')
print('Test 4 PASSED (get_risk always resets -- no accumulate option, always an overwrite)')

first call  grad sum: 2434632.7323671654
second call grad sum: 2434632.7323671654
two separate calls agree -- get_risk is deterministic/idempotent given the same model state
pre-seeded nonzero values are correctly overwritten, not accumulated into
Test 4 PASSED (get_risk always resets -- no accumulate option, always an overwrite)


## Test 5 -- settlement-date branches and the `pay_or_rec` sign

`value_date == payment_date` should realize the full settlement amount as cash with `DF == 1`;
`value_date > payment_date` (fully matured) should give zero PV and zero cash. Also confirms
`PayOrReceive.PAY` flips the sign of PV relative to `PayOrReceive.RECEIVE` for the same product
otherwise unchanged.

In [8]:
SOFR_NAME = 'SOFR-1B'

def set_sofr_fixings(fixing_map: dict) -> None:
    if IndexFixingsManager().exists(SOFR_NAME):
        qfRemoveIndexFixings(SOFR_NAME)
    IndexFixingsManager()._map.setdefault(SOFR_NAME, {})
    dates = [d.ISO() for d in fixing_map]
    values = list(fixing_map.values())
    qfInsertIndexFixing(SOFR_NAME, dates, values)

# valuing on/after payment_date makes the whole accrual period "known", so the engine needs a
# real daily fixing for every business day in [effective_date, termination_date) -- seed a flat
# 4.30% series (matching the curve's own flat level) across that window.
sofr_fixings = {}
d = effective_date
while d < termination_date:
    if hol_conv.isBusinessDay(d):
        sofr_fixings[d] = 0.0430
    d = Date(d + 1)
set_sofr_fixings(sofr_fixings)

63 fixing(s) for SOFR-1B is(are) inserted.


In [9]:
# value_date == payment_date: build a fresh model valued exactly on the payment date
yc_on_payment = qfCreateModel(product.payment_date, 'YIELD_CURVE', build_ifr_data(), build_method_collection)
engine_on_payment = ValuationEngineProductOvernightIndexCompositeCashflow(yc_on_payment, vpc, product, ValuationRequest.PV)
engine_on_payment.calculate_value()
print('value_date == payment_date -> value:', engine_on_payment.value, ' cash:', engine_on_payment.cash, ' df:', engine_on_payment.df_)
assert engine_on_payment.df_ == 1.0
assert engine_on_payment.value == engine_on_payment.cash
assert engine_on_payment.cash != 0.0

# value_date > payment_date: fully matured, nothing left
matured_date = add_period(product.payment_date, Period('1D'), biz_conv, hol_conv)
yc_matured = qfCreateModel(matured_date, 'YIELD_CURVE', build_ifr_data(), build_method_collection)
engine_matured = ValuationEngineProductOvernightIndexCompositeCashflow(yc_matured, vpc, product, ValuationRequest.PV)
engine_matured.calculate_value()
print('value_date > payment_date  -> value:', engine_matured.value, ' cash:', engine_matured.cash)
assert engine_matured.value == 0.0 and engine_matured.cash == 0.0

# PayOrReceive.PAY flips the sign
product_pay = ProductOvernightIndexCompositeCashflow(
    effective_date, TermOrDate(termination_date), PayOrReceive.PAY, sofr_composite_index, spread,
    Currency('USD'), notional,
    payment_business_day_convention=biz_conv, payment_holiday_convention=hol_conv,
)
engine_pay = ValuationEngineProductOvernightIndexCompositeCashflow(yc_usd, vpc, product_pay, ValuationRequest.PV)
engine_pay.calculate_value()
print('RECEIVE value:', float(engine.value.detach()), ' PAY value:', float(engine_pay.value.detach()))
assert abs(float(engine_pay.value.detach()) + float(engine.value.detach())) < 1e-6
print('Test 5 PASSED (settlement-date branches and pay_or_rec sign convention check out)')

value_date == payment_date -> value: tensor(114314.9505, dtype=torch.float64)  cash: 114314.95045180971  df: 1.0
value_date > payment_date  -> value: 0.0  cash: 0.0
RECEIVE value: 111157.35434079105  PAY value: -111157.35434079105
Test 5 PASSED (settlement-date branches and pay_or_rec sign convention check out)


## Summary

In [10]:
print('All ValuationEngineProductOvernightIndexCompositeCashflow tests passed: PV, pv01, get_risk vs finite difference, get_risk always-resets semantics, and settlement/sign branches.')

All ValuationEngineProductOvernightIndexCompositeCashflow tests passed: PV, pv01, get_risk vs finite difference, get_risk always-resets semantics, and settlement/sign branches.
